# ARISTA velocity streams with the CytoBridge API

**Audience.** Researchers reproducing the ARISTA downstream analysis or adapting it to another spatial time-series dataset.

**Prerequisites.** Install the restored `cytobridge-spatial` package in editable mode, then run this notebook from a clone of `cb_reproducibility`. The repository assets are sufficient for this example; on the project server, `CYTOBRIDGE_ARISTA_WORKSPACE` can point to the uploaded full workspace.

**Learning goals.** By the end, you can load the trained ARISTA checkpoint through the public `CytoBridge` API, compute intrinsic/interaction/score velocity components, and reproduce the six t1 stream panels without importing `DeepRUOT` or `vendor/legacy_arista_stack`.


## Outline

1. Verify the installed package and repository paths.
2. Configure portable assets or the server workspace.
3. Load the annotated 52-dimensional model input and trained checkpoint.
4. Run the package-backed velocity workflow.
5. Inspect numerical summaries and rendered panels.
6. Try a different timepoint as an exercise.


In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'downstream_helpers').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'downstream_helpers').is_dir():
    raise RuntimeError('Run this notebook from inside the cb_reproducibility clone.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import CytoBridge
from downstream_helpers.arista_api import (
    AristaVelocityConfig,
    assert_package_only_runtime,
    load_arista_api_context,
    resolve_arista_api_paths,
    run_arista_velocity_t1_streams,
)

assert_package_only_runtime()
print('CytoBridge:', Path(CytoBridge.__file__).resolve())
print('Reproducibility repo:', REPO_ROOT)


## 1. Configure the run

By default the notebook uses the portable model and CSV under `assets/arista/` and `data/arista/`. On the server it automatically uses the uploaded workspace when that directory exists. Set `CYTOBRIDGE_MAX_CELLS` for a fast smoke test; leave it unset for the complete t1 slice.


In [ ]:
server_workspace = Path('/data/cytobridge/projects/CytoBridge-ST-1104/workspace')
workspace_env = os.environ.get('CYTOBRIDGE_ARISTA_WORKSPACE')
workspace_root = Path(workspace_env) if workspace_env else (server_workspace if server_workspace.exists() else None)
max_cells_env = int(os.environ.get('CYTOBRIDGE_MAX_CELLS', '0'))

config = AristaVelocityConfig(
    output_name=os.environ.get('CYTOBRIDGE_OUTPUT_NAME', 'arista_velocity_t1_streams_api'),
    output_dir=os.environ.get('CYTOBRIDGE_OUTPUT_DIR'),
    workspace_root=workspace_root,
    target_timepoint=1.0,
    max_cells=max_cells_env or None,
    random_seed=42,
    device=os.environ.get('CYTOBRIDGE_DEVICE', 'cpu'),
    interaction_m=1024,
    interaction_threshold=1000.0,
)
paths = resolve_arista_api_paths(config)
config, paths


## 2. Load data and model through the package

The original checkpoint expects 52 model features: two aligned spatial dimensions followed by 50 latent dimensions. The helper validates this contract before inference. The 2,000-gene H5AD is the upstream biological object; its expression matrix must not be passed directly to this 52-dimensional checkpoint.


In [ ]:
context = load_arista_api_context(config)
assert_package_only_runtime()

print('Input table:', context['df'].shape)
print('Model feature dimension:', context['dim'])
print('Observed times:', sorted(context['df']['samples'].unique().tolist()))
print('Model class:', type(context['loaded'].model).__name__)
print('Components:', context['loaded'].model.components)


## 3. Compute and render the t1 velocity decomposition

All numerical work below is delegated to `CytoBridge.tl.compute_velocity_components`; plotting is delegated to `CytoBridge.pl.plot_velocity_component`. The reproducibility helper only resolves paths, validates dimensions, and writes a manifest.


In [ ]:
result = run_arista_velocity_t1_streams(config)
assert_package_only_runtime()
result


In [ ]:
summary = pd.read_csv(result.component_summary_csv)
summary


In [ ]:
for panel_name, figure_path in result.figure_paths.items():
    print(panel_name, figure_path.name)
    display(SVG(filename=str(figure_path)))


## Pitfall and extension

**Common mistake:** passing `adata.X` from `ARTISTA_after_pp_with_ae_and_center.h5ad` directly into the trained model. That matrix has 2,000 genes, whereas this checkpoint was trained on the aligned 52-dimensional CSV. Use the model-input CSV for checkpoint inference; use the H5AD for the future end-to-end preprocessing retrain.

**Extension:** change `target_timepoint` below and compare whether the intrinsic and interaction fields reinforce or oppose each other. Use a new `output_name` so results are not overwritten.


## Exercise

Create a timepoint-2 configuration. First run it with 256 cells, then remove the cap for the complete slice. Predict which velocity component will change most before looking at the summary table.


In [ ]:
exercise_config = replace(
    config,
    output_name='arista_velocity_t2_streams_api_exercise',
    target_timepoint=2.0,
    max_cells=256,
)
# Uncomment after making a prediction:
# exercise_result = run_arista_velocity_t1_streams(exercise_config)
exercise_config
